In [ ]:
from langchain.tools import Tool
def calculate(expression:str) -> str:
    try:
        result = eval(expression)
        return f"计算结果:{result}"
    except Exception as e:
        return f"计算错误:{str(e)}"
calculator_tool = Tool(
    name='Calculator',
    func=calculate,
    description="""
    用于数学计算。
    输入：数学表达式字符串，如 "2 + 2" 或 "sqrt(16)"
    输出：计算结果
    何时使用：当需要进行精确数学计算时
    """
)

In [ ]:
from langchain.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun
search_tool = Tool(
    name='WebSearch',
    func=search.run,
    description='搜索互联网获取最新信息。适用于需要实时数据的场景。'
)

In [ ]:
def python_executor(code:str) -> str:
    import io
    import sys
    from contextlib import redirect_stdout
    f = io.StringIO()
    try:
        with redirect_stdout(f):
            exec(code)
        return f.getvalue()
    except Exception as e:
        return f"错误:{str(e)}"
code_tool = Tool(
    name='PythonEXecutor',
    func=python_executor,
    description="执行Python代码。适用于数据处理、计算、可视化等任务。"
)

In [ ]:
import requests
def weather_api(city:str) -> str:
    response = requests.get(f"hhtp://api.weather.com/{city}")
    return response.json()
weather_tool = Tool(
    name="WeatherAPI",
    func=weather_api,
    description="查询指定城市的天气信息。输入城市名，返回温度、湿度等信息。"
)

In [ ]:
import sqlite3
def query_database(sql:str) -> str:
    conn = sqlite3.connect('my_database.db')
    cursor = conn.cursor()
    try:
        cursor.execute(sql)
        results = cursor.fetchall()
        return str(results)
    except Exception as e:
        return f"查询错误:{str(e)}"
    finally:
        conn.close()
db_tool = Tool(
    name="DatabaseQuery",
    func=query_database,
    description="执行SQL查询。适用于需要从数据库获取数据的场景。"
)

In [ ]:
from langchain.agents import create_react_agent,AgentExecutor
from langchain_openai import ChatOpenAI
tools = [
    search_tool,
    code_tool,
    db_tool,
]
llm = ChatOpenAI(model='gpt-4',temperature=0)
agent = create_react_agent(llm,tools)
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)
result = agent_executor.invoke({
    "input": """
    帮我分析一下：
    1. 从数据库中查询2024年的销售数据
    2. 搜索行业平均增长率
    3. 用Python计算我们的增长率
    4. 对比并给出结论
    """
})

In [2]:
from multiprocessing import process
def safe_tool(input_data):
    try:
        result = process(input_data)
        return result
    except Exception as e:
        return f"工具执行失败:{str(e)}。建议：尝试其他方法或简化输入。"

In [1]:
import logging
def logged_tool(input_data):
    logging.info(f"工具被调用,输入:{input_data}")
    result = execute(input_data)
    logging.info(f"工具执行完成，输出：{result}")
    return result

In [ ]:
from langchain_classic.agents import create_react_agent,AgentExecutor
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
import os
os.environ['DEEPSEEK_API_KEY'] = "sk-API KEY"
react_prompt = PromptTemplate.from_template("""
你是一个AI助手,通过推理和行动来解决问题。

可用工具：
{tools}

使用以下格式：

Question: 需要回答的问题
Thought: 你应该思考该做什么
Action: 要采取的行动，必须是[{tool_names}]之一
Action Input: 行动的输入
Observation: 行动的结果
... (这个Thought/Action/Action Input/Observation可以重复N次)
Thought: 我现在知道最终答案了
Final Answer: 原始问题的最终答案

开始！

Question: {input}
Thought: {agent_scratchpad}
"""
)
def search(query:str) -> str:
    if "金牌" in query:
        return "2024年巴黎奥运会美国队40枚金牌第一"
    elif "首都" in query:
        return "美国首都是华盛顿D.C."
    return "未找到相关信息"
search_tool = Tool(
    name="Search",
    func=search,
    description="搜索互联网信息"
)
llm = ChatDeepSeek(model="deepseek-chat",temperature=0)
agent = create_react_agent(llm=llm,tools=[search_tool],prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,tools=[search_tool],verbose=True,max_iterations=5
)
try:
    result = agent_executor.invoke({
         "input":"2024年哪个国家赢得了最多的奥运金牌？这个国家的首都是哪里？"
    })
except Exception as e:
   print("ERROR:", type(e).__name__, str(e))



> Entering new AgentExecutor chain...
Thought: 我需要先查2024年奥运会金牌榜，找出金牌最多的国家，然后确认其首都。  
Action: Search  
Action Input: 2024年奥运会金牌榜 哪个国家金牌最多  2024年巴黎奥运会美国队40枚金牌第一Thought: 搜索结果已经明确：2024年巴黎奥运会金牌最多的是美国（40枚金牌）。接下来需要确认美国的首都。  
Action: Search  
Action Input: 美国的首都是哪里  美国首都是华盛顿D.C.Thought: 我已经知道2024年巴黎奥运会金牌最多的国家是美国，并且美国的首都是华盛顿D.C.，可以给出最终答案了。  
Final Answer: 2024年巴黎奥运会赢得金牌最多的是美国（40枚金牌），美国的首都是华盛顿哥伦比亚特区（Washington, D.C.）。

> Finished chain.


难点一：无限循环与任务卡死
Agent可能陷入死循环：
Thought: 我需要搜索信息
Action: Search("2024 AI")
Observation: [搜索结果]

Thought: 我需要搜索更详细的信息
Action: Search("2024 AI详细")
Observation: [搜索结果]

Thought: 我需要搜索更详细的信息
Action: Search("2024 AI更详细")
...（永远循环）

In [ ]:
#解决方案一：设置最大迭代次数
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    max_iterations=10,
    max_execution_time=60,
    early_stopping_method="generate"
)
#方案二：优化提示词，明确终止条件
prompt = """
你必须在5步内完成任务。
如果5步内无法完成,请给出"基于现有信息的最佳答案"。

当前已经执行了{iterations}步。
"""
#方案三：实现智能终止判断
def should_continue(agent_state):
    last_3_actions = agent_state.history[-3:]
    if len(set(last_3_actions)) == 1:
        return False
    if agent_state.iterations > 5 and not has_new_info(agent_state):
        return False
    return True

难点二：工具选择错误
Agent选择了错误的工具：
任务："计算2024年有多少天"

错误选择：
Action: WebSearch("2024年有多少天")  ❌ 不需要搜索

正确选择：
Action: Calculator("365 if not is_leap_year(2024) else 366")  ✅

In [ ]:
#方案一：改进工具描述
calculator_tool = Tool(
    name="Calculator",
    description="""
    用于精确的数学计算。
    
    适用场景：
    - 算术运算（加减乘除）
    - 数学函数（平方、开方、三角函数）
    - 逻辑判断（比较大小）
    
    输入格式：Python数学表达式字符串
    示例："2 + 2", "sqrt(16)", "2024 % 4 == 0"
    
    不适用场景：
    - 需要最新数据（用Search）
    - 需要上下文信息（用Memory）
    """
)
#方案二:添加工具使用示例
prompt = """
工具使用示例：

问题："今天的天气如何？"
正确：Action: WeatherAPI("北京")
错误：Action: Calculator("天气")  # 天气不是数学问题

问题："2的100次方是多少？"
正确：Action: Calculator("2**100")
错误：Action: Search("2的100次方")  # 不需要搜索

现在请处理：{input}
"""
#方案三：实现工具推荐系统
def recommend_tool(task_description,available_tools):
    analysis = llm.invoke(f"""
    任务：{task_description}
    
    请分析这个任务需要：
    1. 实时数据吗？→ 需要Search
    2. 数学计算吗？→ 需要Calculator
    3. 历史信息吗？→ 需要Memory
    
    推荐工具（JSON格式）：
    {{"recommended": ["tool1", "tool2"], "reason": "..."}}
    """)

难点三：上下文窗口溢出
Prompt + 历史对话 + 工具描述 + 中间结果 = 超过Token限制

GPT-4: 128K tokens
Claude: 200K tokens

一次复杂任务可能消耗50K+ tokens
→ 超出限制或成本过高

In [ ]:
#方案一：智能压缩上下文
from langchain.memory import ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_list=4000,
    moving_summary_buffer="",
)
#方案二：分层记忆
class HierarchicalMemory:
    def init(self):
        self.recent_memory = []
        self.mid_term_memory = []
        self.long_term_memory = vectorstore
    def add(self,message):
        self.recent_memory.append(message)
        if len(self.recent_memory) > 3:
            old_message = self.recent_memory.pop(0)
            key_info = extract_key_info(old_message)
            self.mid_term_memory.append(key_info)
        if len(self.mid_term_memory) > 10:
            old_info = self.mid_term_memory.pop(0)
            self.long_term_memory.add_texts([old_info])
    def get_context(self,query):
        recent = "\n".join(self.recent_memory)
        mid = "\n".join(self.mid_term_memory)
        relevant = self.long_term_memory.similarity_search(query,k=2)
        return f"{recent}\n\n相关历史：{mid}\n{relevant}"
#方案3：动态工具加载
all_tools = {
    "search":search_tool,
    "calculator":calc_tool,
    "database":db_tool,
    "api":api_tool,
}
def select_tools(task):
    if "搜索" in task or "查询" in task:
        return [all_tools["search"]]
    elif "计算" in task:
        return [all_tools["calculator"]]
    return list(all_tools.values())[:5]
tools = select_tools(user_input)
agent = create_agent(llm,tools)

难点四：错误处理与鲁棒性
问题描述
各种错误会导致Agent崩溃：
- 工具调用失败
- API超时
- 返回格式错误
- LLM输出异常

In [ ]:
#方案1：工具层面的错误处理
def robust_tool(func):
    def wrapper(*args,**kwargs):
        max_retries = 3
        for attempt in range(max_retries):
            try:
                result = func(*args,**kwargs)
                return {
                    "success":True,
                    "data":result
                }
            except TimeoutError:
                if attempt == max_retries - 1:
                    return {
                        "success":False,
                        "error":"工具执行超时，请尝试简化输入或使用其他工具"
                    }
                time.sleep(2 ** attempt)
            except Exception as e:
                return {
                    "success":False,
                    "error":f"工具执行失败：{str(e)}"}
    return wrapper
@robust_tool
def search_tool(query):
    return search_engine.query(query)
#方案2：Agent层面的降级策略
class RobustAgent:
    def execute(self,task):
        try:
            return self.full_execution(task)
        except AgentError:
            return self.simplified_execution(task)
        except Exception:
            return self.fallback_execution(task)
    def full_execution(self,task):
        return self.react_loop(task)
    def simplified_execution(self,task):
        tool = self.select_best_tool(task)
        result = tool.run(task)
        return self.llm.summarize(result)
    def fallback_execution(self,task):
        return self.llm.invoke(f"请基于你的知识回答：{task}") 
# 方案3：实时监控与告警
import logging
class AgentMonitor:
    def init(self):
        self.metrics = {
            'total_tasks':0,
            'success':0,
            'failures':0,
            'avg_time':0,
        }
    def log_execution(self,task,result,duration):
        self.metrics["total_tasks"] += 1
        if result["success"]:
            self.metrics["success"] += 1
        else:
            self.metrics["failures"] += 1
            logging.error(f"Task failed:{task},Error:{result['error']}")
        self.metrics["avg_time"] = (
            self.metrics["avg_time"] * (self.metrics["total_tasks"] - 1) + duration
        ) / self.metrics["total_tasks"]
        failuer_rate = self.metrics["failures"] / self.metrics["total_tasks"]
        if failure_rate > 0.3:
            send_alert(f"Agent failure rate: {failure_rate:.2%}")

难点五：成本控制
一次复杂任务：
10轮ReAct循环
每轮2K tokens输入 + 500 tokens输出
总计：(2000+500) * 10 = 25K tokens

GPT-4价格（假设）：
输入：$0.03 / 1K tokens
输出：$0.06 / 1K tokens

单次任务成本：
输入：20K * 0.03 = $0.60
输出：5K * 0.06 = $0.30
总计：$0.90

如果每天1000个任务 = $900/天 = $27,000/月

In [ ]:
#方案1：模型分级使用
class CostOptimizerAgent:
    def init(self):
        self.models = {
            "cheap":ChatOpenAI(model="gpt-3.5-turbo"),
            "standard":ChatOpenAI(model="gpt-4"),
            "premium":ChatOpenAI(model="gpt-4-turbo"),
        }
    def select_model(self,task_complexity):
        if task_comlexity < 3:
            return self.models["cheap"]
        elif task_complexity < 7:
            return self.models["standard"]
        else:
            return self.models["premium"]
    def estimate_complexity(self,task):
        factors = {
            "num_steps":len(parse_steps(task)) * 2,
            "need_tools":3 if requires_tools(task) else 0,
            "domain_specific":2 if is_specialized(task) else 0,
        }
        return sum(factors.values())
#方案2：缓存机制
from functools import lru_cache
import hashlib
class CacheAgent:
    def init(self):
        self.cache = {}
    def execute(self,task):
        task_hash = hashlib.md5(task.encode()).hexdigest()
        if task_hash in self.cache:
            logging.info(f"Cache hit for task: {task[:50]}...")
            return self.cache[task_hash]
        result = self.agent.run(task)
        self.cache[task_hash] = result
        return result
#方案3：批处理
def batch_process(tasks):
    groups = group_similar_tasks(tasks)
    result = []
    for group in groups:
        batch_prompt = f"""
        请处理以下{len(group)}个相似任务：
        {"\n".join([f"{i+1}. {t}" for i, t in enumerate(group)])}
        
        请输出JSON格式的结果列表。
        """
    return results
#方案4：设置预算限制
class BudgetControlledAgent:
    def init(self,daily_budget_usd=10):
        self.daily_budget = daily_budget_usd
        self.today_spent = 0
        self.task_count = 0
    def execute(self,task):
        estimated_cost = self.estimate_cost(task)
        if self.today_spent + estimated_cost > self.daily_budget:
            return {
                "success": False,
                "message": f"预算不足。今日已用${self.today_spent:.2f}，预算${self.daily_budget}"
            }
        result = self.agent.run(task)
        actual_cost = self.calculate_cost(result.tokens_used)
        self.today_spent += actual_cost
        self.task_count += 1
        logging.info(f"Task {self.task_count}: Cost ${actual_cost:.4f}, Total ${self.today_spent:.2f}")
        return result